# 03 — Query Iceberg Tables

**Pre-requisites:**
- Run `01_csv_to_iceberg.ipynb` to populate `iceberg_catalog.my_database.orders`
- Run `02_streaming_to_iceberg.ipynb` (with producer) to populate `iceberg_catalog.my_database.user_events`

**What this notebook covers:**
- Spark SQL queries on both tables
- DataFrame API with aggregations and joins
- Iceberg time-travel and snapshot inspection

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Spark-Developer").master("local[*]").getOrCreate()

spark

:: loading settings :: url = jar:file:/Users/saketkumar/Documents/github/helper/spark-dev/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/saketkumar/.ivy2.5.2/cache
The jars for the packages stored in: /Users/saketkumar/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-43e3e4cb-dd85-49be-bd13-521866e3f013;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in central
:: resolution report :: resolve 51ms :: artifacts dl 2ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-------------------------

## Discover tables

In [2]:
spark.sql("SHOW TABLES IN iceberg_catalog.my_database").show(truncate=False)

+-----------+-----------+-----------+
|namespace  |tableName  |isTemporary|
+-----------+-----------+-----------+
|my_database|user_events|false      |
|my_database|orders     |false      |
+-----------+-----------+-----------+



---
## Table 1 — `orders` (batch / CSV source)

In [3]:
# Full table scan — SQL
spark.sql("SELECT * FROM iceberg_catalog.my_database.orders ORDER BY order_id").show(truncate=False)

+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pending  |
|7       |Grace Kim    |USB Hub     |1       |49.99     |2024-01-21|completed|
|8       |Henry Wilson |Laptop Stand|2       |59.99     |2024-01-22|shipped  |
|9       |Iris Chen    |SSD Drive   |1       |119.99    |2024-01-23|completed|
|10      |James Davis  |Microphone  |1       |199.99

In [4]:
# Revenue by status — SQL
spark.sql("""
    SELECT
        status,
        COUNT(*)                             AS orders,
        SUM(quantity)                        AS units_sold,
        ROUND(SUM(quantity * unit_price), 2) AS revenue
    FROM iceberg_catalog.my_database.orders
    GROUP BY status
    ORDER BY revenue DESC
""").show()

+---------+------+----------+-------+
|   status|orders|units_sold|revenue|
+---------+------+----------+-------+
|completed|     5|         6|1929.94|
|  shipped|     2|         5| 569.95|
|  pending|     3|         4| 459.96|
+---------+------+----------+-------+



In [5]:
# Top products by revenue — DataFrame API
orders_df = spark.table("iceberg_catalog.my_database.orders")

(
    orders_df
    .withColumn("revenue", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .groupBy("product")
    .agg(
        F.sum("revenue").alias("total_revenue"),
        F.sum("quantity").alias("total_units"),
        F.countDistinct("customer_name").alias("unique_customers"),
    )
    .orderBy(F.desc("total_revenue"))
    .show(truncate=False)
)

+------------+-------------+-----------+----------------+
|product     |total_revenue|total_units|unique_customers|
+------------+-------------+-----------+----------------+
|Laptop      |1299.99      |1          |1               |
|Headphones  |449.97       |3          |1               |
|Monitor     |399.99       |1          |1               |
|Microphone  |199.99       |1          |1               |
|Webcam      |179.98       |2          |1               |
|SSD Drive   |119.99       |1          |1               |
|Laptop Stand|119.98       |2          |1               |
|Keyboard    |79.99        |1          |1               |
|Mouse       |59.98        |2          |1               |
|USB Hub     |49.99        |1          |1               |
+------------+-------------+-----------+----------------+



---
## Table 2 — `user_events` (streaming / JSON source)

In [6]:
# Full table scan — SQL
spark.sql("""
    SELECT * FROM iceberg_catalog.my_database.user_events
    ORDER BY timestamp DESC
    LIMIT 20
""").show(truncate=False)

+------------------------------------+--------+-----------+---------+--------------------------------+----------+
|event_id                            |user_id |event_type |page     |timestamp                       |session_id|
+------------------------------------+--------+-----------+---------+--------------------------------+----------+
|0920ac28-da63-429b-948c-d78a621fae15|user_020|purchase   |/search  |2026-04-25T05:36:35.027222+00:00|2d498a29  |
|a059ee22-e3ea-490d-ab94-4891042dc8a1|user_005|search     |/orders  |2026-04-25T05:36:35.027208+00:00|daa86615  |
|80fc8f3c-da46-4158-b7d9-c8c5dbb136a3|user_007|purchase   |/home    |2026-04-25T05:36:25.019564+00:00|9fcab04b  |
|036c5c21-fae7-471f-af96-dc6fdaee9a6c|user_019|search     |/cart    |2026-04-25T05:36:25.019485+00:00|06f4efd2  |
|9da28a59-2ee6-43b5-9698-5336deb2c519|user_013|purchase   |/home    |2026-04-25T05:36:15.012963+00:00|576683fc  |
|b6d519ee-9efe-4e20-a3da-f8945809dcc8|user_014|purchase   |/cart    |2026-04-25T05:36:15

In [7]:
# Event breakdown — SQL
spark.sql("""
    SELECT
        event_type,
        COUNT(*)                    AS event_count,
        COUNT(DISTINCT user_id)     AS unique_users,
        COUNT(DISTINCT session_id)  AS unique_sessions
    FROM iceberg_catalog.my_database.user_events
    GROUP BY event_type
    ORDER BY event_count DESC
""").show()

+-----------+-----------+------------+---------------+
| event_type|event_count|unique_users|unique_sessions|
+-----------+-----------+------------+---------------+
|   purchase|         38|          18|             38|
|     search|         34|          16|             34|
|     logout|         32|          15|             32|
|add_to_cart|         28|          16|             28|
|      click|         26|          16|             26|
|  page_view|         16|          11|             16|
+-----------+-----------+------------+---------------+



In [8]:
# Top active users — DataFrame API
events_df = spark.table("iceberg_catalog.my_database.user_events")

(
    events_df
    .groupBy("user_id")
    .agg(
        F.count("*").alias("total_events"),
        F.countDistinct("event_type").alias("distinct_event_types"),
        F.countDistinct("page").alias("pages_visited"),
    )
    .orderBy(F.desc("total_events"))
    .show(10)
)

+--------+------------+--------------------+-------------+
| user_id|total_events|distinct_event_types|pages_visited|
+--------+------------+--------------------+-------------+
|user_009|          15|                   5|            6|
|user_007|          13|                   6|            7|
|user_014|          11|                   4|            6|
|user_003|          11|                   4|            6|
|user_005|          11|                   5|            5|
|user_018|          10|                   5|            7|
|user_016|          10|                   6|            6|
|user_017|           9|                   4|            6|
|user_011|           9|                   5|            5|
|user_019|           9|                   5|            4|
+--------+------------+--------------------+-------------+
only showing top 10 rows


In [9]:
# Most visited pages — DataFrame API
(
    events_df
    .groupBy("page")
    .agg(F.count("*").alias("visits"))
    .orderBy(F.desc("visits"))
    .show()
)

+---------+------+
|     page|visits|
+---------+------+
|    /cart|    28|
|/products|    27|
|    /home|    27|
| /profile|    25|
|/checkout|    24|
|  /orders|    22|
|  /search|    21|
+---------+------+



---
## Cross-table analysis

Find users who placed orders **and** had streaming events  
(works once `customer_name` and `user_id` overlap, e.g. `user_001` etc.)

In [10]:
# Purchase events alongside order revenue per customer — SQL
spark.sql("""
    SELECT
        e.user_id,
        COUNT(e.event_id)                        AS total_events,
        SUM(CASE WHEN e.event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_events,
        COUNT(DISTINCT e.page)                   AS pages_visited
    FROM iceberg_catalog.my_database.user_events e
    GROUP BY e.user_id
    ORDER BY purchase_events DESC, total_events DESC
""").show(10)

+--------+------------+---------------+-------------+
| user_id|total_events|purchase_events|pages_visited|
+--------+------------+---------------+-------------+
|user_009|          15|              5|            6|
|user_003|          11|              4|            6|
|user_011|           9|              4|            5|
|user_010|           7|              4|            4|
|user_017|           9|              3|            6|
|user_007|          13|              2|            7|
|user_005|          11|              2|            5|
|user_018|          10|              2|            7|
|user_016|          10|              2|            6|
|user_019|           9|              2|            4|
+--------+------------+---------------+-------------+
only showing top 10 rows


---
## Iceberg metadata & time travel

In [11]:
# Snapshots for orders
print("=== orders snapshots ===")
spark.sql("""
    SELECT snapshot_id, committed_at, operation, summary
    FROM iceberg_catalog.my_database.orders.snapshots
""").show(truncate=False)

# Snapshots for user_events
print("=== user_events snapshots ===")
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM iceberg_catalog.my_database.user_events.snapshots
    ORDER BY committed_at
""").show(20, truncate=False)

=== orders snapshots ===
+-------------------+-----------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |committed_at           |operation|summary                                                                                                                                                                                                                                                                                                                                                                                                            

In [12]:
# Time travel — read orders at a specific snapshot
snapshots = spark.sql("""
    SELECT snapshot_id FROM iceberg_catalog.my_database.orders.snapshots ORDER BY committed_at
""").collect()

if snapshots:
    first_snapshot = snapshots[0]["snapshot_id"]
    print(f"Reading orders at snapshot: {first_snapshot}")
    spark.read \
         .option("snapshot-id", first_snapshot) \
         .table("iceberg_catalog.my_database.orders") \
         .show(truncate=False)
else:
    print("No snapshots found — run 01_csv_to_iceberg.ipynb first.")

Reading orders at snapshot: 4114315725319939862
+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pending  |
|7       |Grace Kim    |USB Hub     |1       |49.99     |2024-01-21|completed|
|8       |Henry Wilson |Laptop Stand|2       |59.99     |2024-01-22|shipped  |
|9       |Iris Chen    |SSD Drive   |1       |119.99    |2024-01-23|completed|
|10 

In [13]:
# Data files layout — Iceberg metadata tables
print("=== orders data files ===")
spark.sql("""
    SELECT file_path, file_format, record_count, file_size_in_bytes
    FROM iceberg_catalog.my_database.orders.files
""").show(truncate=False)

print("=== user_events data files ===")
spark.sql("""
    SELECT file_path, file_format, record_count, file_size_in_bytes
    FROM iceberg_catalog.my_database.user_events.files
""").show(truncate=False)

=== orders data files ===
+-----------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+
|file_path                                                                                                        |file_format|record_count|file_size_in_bytes|
+-----------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+
|.tmp/local_iceberg_warehouse/my_database/orders/data/00000-1-c556d7ef-f7af-46cb-bf29-a402ba0dd98a-0-00001.parquet|PARQUET    |10          |2520              |
+-----------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+

=== user_events data files ===
+-------------------------------------------------------------------------------------------------------------------------+-----------+-------